# 07 — XGBoost v2 (Tight Tuning with Fold-Level Early Stopping)

**Why this notebook exists.** The first XGBoost pass (notebook 04) ended at CV MSE ≈ 222.8 — worse than GradientBoosting (183.8). Root causes:
1. `RandomizedSearchCV` with no early stopping wasted budget on under-trained configs.
2. Tuning was 3-fold but final eval was 5-fold (slight mismatch).
3. The chosen `learning_rate ≈ 0.016` × `n_estimators = 977` ⇒ effective shrinkage too low, model under-converged.

**Strategy here.**
* Hand-rolled 5-fold CV — same seed as everything else (so OOFs stay blendable).
* Inside each fold, split a 10% inner validation set for **`early_stopping_rounds=50`**.
* Random search over **60 configurations** with a focused, well-defined parameter space.
* `log1p` target + `KFoldTargetEncoder` (Module-7 custom transformer) reused.
* For each config we report mean **fold-best-MSE** ± std → pick the lowest mean.
* Refit on full train with the best config + final OOF.

Outputs:
* `outputs/oof_XGBoostV2.npy` — replaces `oof_XGBoost.npy` in the blend
* `outputs/submission_xgboost_v2.csv`
* `outputs/xgb_v2_best_params.json`
* `outputs/xgb_v2_search_log.csv` — every config + score, for the paper.

All techniques are course-allowed: XGBoost + RandomizedSearch + KFold CV + Pipeline + custom transformer + early stopping (Module 10/11).

## 1. Setup & Data

In [ ]:
import warnings, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
N_SPLITS = 5

OUT_DIR = Path('../outputs'); OUT_DIR.mkdir(exist_ok=True, parents=True)
print('OUT_DIR =', OUT_DIR.resolve())

OUT_DIR = C:\pprojelerim\Airbnb-K353\outputs


In [ ]:
train_df = pd.read_parquet(OUT_DIR / 'train_features.parquet')
test_df  = pd.read_parquet(OUT_DIR / 'test_features.parquet')
TARGET = 'NumReserveDays2016Q3'; ID = 'PropertyID'
y = train_df[TARGET].astype(float).values
X = train_df.drop(columns=[TARGET, ID]).reset_index(drop=True)
X_test = test_df.drop(columns=[ID]).reindex(columns=X.columns).reset_index(drop=True)
test_ids = test_df[ID].values

cat_all = X.select_dtypes(exclude='number').columns.tolist()
card    = {c: X[c].nunique(dropna=False) for c in cat_all}
cat_high = [c for c, n in card.items() if n > 15]
cat_low  = [c for c, n in card.items() if n <= 15]
num_cols = X.select_dtypes(include='number').columns.tolist()
print(f'num={len(num_cols)}  cat_low={cat_low}  cat_high={cat_high}')

num=130  cat_low=['ListingType', 'MetropolitanStatisticalArea', 'CancellationPolicy']  cat_high=['PropertyType', 'Neighborhood']


## 2. K-Fold Target Encoder (Module-7 custom transformer)

In [ ]:
class KFoldTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols, n_splits=5, smoothing=20.0, random_state=42):
        self.cols = cols; self.n_splits = n_splits
        self.smoothing = smoothing; self.random_state = random_state

    def _smap(self, x, y):
        st = pd.DataFrame({'c': x, 'y': y}).groupby('c')['y'].agg(['mean', 'count'])
        return ((st['count'] * st['mean'] + self.smoothing * self.global_mean_)
                / (st['count'] + self.smoothing)).to_dict()

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        self.global_mean_ = float(y.mean())
        self.maps_ = {c: self._smap(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return self

    def transform(self, X):
        Xo = X.copy()
        for c in self.cols:
            Xo[c] = (X[c].astype(str).fillna('__nan__').map(self.maps_[c])
                       .fillna(self.global_mean_).astype('float32'))
        return Xo

    def fit_transform(self, X, y=None, **kw):
        y = np.asarray(y, dtype=float); self.global_mean_ = float(y.mean())
        Xo = X.copy()
        for c in self.cols: Xo[c] = np.full(len(X), self.global_mean_, dtype='float32')
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        for tr, va in kf.split(X):
            for c in self.cols:
                m = self._smap(X[c].astype(str).fillna('__nan__').iloc[tr], y[tr])
                Xo.iloc[va, Xo.columns.get_loc(c)] = (
                    X[c].astype(str).fillna('__nan__').iloc[va].map(m)
                       .fillna(self.global_mean_).astype('float32').values)
        self.maps_ = {c: self._smap(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return Xo

    def get_feature_names_out(self, input_features=None):
        return np.array(input_features if input_features is not None else self.cols)

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
        ('low', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
            ('oh',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_low),
        ('high', Pipeline([
            ('te', KFoldTargetEncoder(cols=cat_high, n_splits=5, smoothing=20, random_state=RANDOM_STATE)),
        ]), cat_high),
    ])

## 3. Fold-Level Early-Stopping Evaluator
Each candidate hyperparameter config is evaluated like this:
1. Run a 5-fold CV with the *same* outer split as our other notebooks.
2. **Inside each fold:** carve a 10% inner-validation slice from the training portion. Fit XGBoost with `early_stopping_rounds=50` watching that inner slice. Use the resulting model to predict the outer-val slice → fold MSE.
3. Return mean ± std over folds.
This makes `n_estimators` adaptive: each fold picks the best iteration on its own.

In [ ]:
outer_kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
outer_splits = list(outer_kf.split(X))   # cache so all configs see the same folds

def evaluate_config(params, return_oof=False, return_models=False, verbose=False):
    """Run 5-fold CV with fold-level early stopping. Returns mean MSE, std, best_iters.
    Optionally returns OOF predictions and the trained models for later refit/inference."""
    fold_mses, best_iters, models = [], [], []
    oof = np.zeros(len(y)) if return_oof else None

    for fold, (tr, va) in enumerate(outer_splits):
        pp = make_preprocessor()
        X_tr_full = pp.fit_transform(X.iloc[tr], y[tr])
        X_va      = pp.transform(X.iloc[va])

        # inner split for early stopping
        X_tr, X_in, y_tr, y_in = train_test_split(
            X_tr_full, y[tr], test_size=0.10, random_state=RANDOM_STATE + fold,
        )
        y_tr_log = np.log1p(y_tr); y_in_log = np.log1p(y_in)

        model = XGBRegressor(
            **params,
            objective='reg:squarederror',
            tree_method='hist',
            random_state=RANDOM_STATE + fold,
            n_jobs=-1,
            early_stopping_rounds=50,
            eval_metric='rmse',
        )
        model.fit(X_tr, y_tr_log, eval_set=[(X_in, y_in_log)], verbose=False)
        best_iter = model.best_iteration

        # predict outer val, invert log1p, clip to [0, 92]
        pred = np.clip(np.expm1(model.predict(X_va, iteration_range=(0, best_iter + 1))), 0, 92)
        mse  = mean_squared_error(y[va], pred)

        fold_mses.append(mse); best_iters.append(best_iter)
        if return_oof: oof[va] = pred
        if return_models: models.append((model, pp, best_iter))
        if verbose: print(f'    fold {fold+1}: MSE={mse:.3f}, best_iter={best_iter}')

    return {
        'mse_mean': float(np.mean(fold_mses)),
        'mse_std':  float(np.std(fold_mses)),
        'best_iters': best_iters,
        'fold_mses': fold_mses,
        'oof': oof,
        'models': models,
    }

## 4. Sanity-Check Config (Sensible Defaults)
Confirms the pipeline works and gives us a sane baseline before any search.

In [ ]:
default_cfg = dict(
    n_estimators=3000,           # large; early stopping picks the real count
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=4,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
)
t0 = time.time()
res0 = evaluate_config(default_cfg, verbose=True)
print(f"\nDefault XGB | MSE = {res0['mse_mean']:.3f} ± {res0['mse_std']:.3f} "
      f"| iters = {res0['best_iters']} | {time.time()-t0:.1f}s")

    fold 1: MSE=213.634, best_iter=157


    fold 2: MSE=239.859, best_iter=190


    fold 3: MSE=228.116, best_iter=186


    fold 4: MSE=220.587, best_iter=141


    fold 5: MSE=226.887, best_iter=146

Default XGB | MSE = 225.817 ± 8.711 | iters = [157, 190, 186, 141, 146] | 10.0s


## 5. Random Search — 60 configurations
Search space focused on the ranges that matter for tabular boosting. Learning rates are sampled log-uniform but biased toward the practical 0.03-0.08 sweet spot.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

def sample_config():
    return dict(
        n_estimators     = 3000,   # cap; early-stop will trim
        learning_rate    = float(np.exp(rng.uniform(np.log(0.02), np.log(0.10)))),
        max_depth        = int(rng.integers(4, 9)),       # 4..8
        min_child_weight = int(rng.integers(1, 12)),      # 1..11
        subsample        = float(rng.uniform(0.65, 0.95)),
        colsample_bytree = float(rng.uniform(0.65, 0.95)),
        gamma            = float(rng.uniform(0.0, 0.4)),
        reg_alpha        = float(np.exp(rng.uniform(np.log(1e-3), np.log(0.5)))),
        reg_lambda       = float(np.exp(rng.uniform(np.log(0.5),  np.log(5.0)))),
    )

N_ITER = 60
log_rows = []
best_score = res0['mse_mean']
best_cfg   = default_cfg.copy()

t_search = time.time()
for i in range(N_ITER):
    cfg = sample_config()
    t0 = time.time()
    res = evaluate_config(cfg)
    elapsed = time.time() - t0
    log_rows.append({**cfg, 'mse_mean': res['mse_mean'], 'mse_std': res['mse_std'],
                     'avg_best_iter': int(np.mean(res['best_iters'])), 'seconds': elapsed})
    flag = ''
    if res['mse_mean'] < best_score:
        best_score = res['mse_mean']
        best_cfg   = cfg.copy()
        flag = '  ←  NEW BEST'
    print(f"[{i+1:2d}/{N_ITER}] MSE = {res['mse_mean']:7.3f} ± {res['mse_std']:5.3f} "
          f"| iters≈{int(np.mean(res['best_iters'])):4d} | lr={cfg['learning_rate']:.4f} "
          f"depth={cfg['max_depth']} | {elapsed:5.1f}s{flag}")

print(f'\nSearch finished in {(time.time()-t_search)/60:.1f} min')
print(f'Best CV MSE: {best_score:.3f}')
print('Best config:', best_cfg)

log_df = pd.DataFrame(log_rows).sort_values('mse_mean').reset_index(drop=True)
log_df.to_csv(OUT_DIR / 'xgb_v2_search_log.csv', index=False)
with open(OUT_DIR / 'xgb_v2_best_params.json', 'w') as f:
    json.dump(best_cfg, f, indent=2)
log_df.head(10)

[ 1/60] MSE = 227.117 ± 7.758 | iters≈ 159 | lr=0.0695 depth=7 |  10.5s


[ 2/60] MSE = 227.374 ± 8.257 | iters≈ 127 | lr=0.0709 depth=6 |   6.9s


[ 3/60] MSE = 225.215 ± 7.026 | iters≈ 228 | lr=0.0408 depth=6 |   9.0s  ←  NEW BEST


[ 4/60] MSE = 232.360 ± 7.659 | iters≈ 428 | lr=0.0354 depth=4 |   8.8s


[ 5/60] MSE = 223.729 ± 6.376 | iters≈ 377 | lr=0.0256 depth=7 |  17.1s  ←  NEW BEST


...

[57/60] MSE = 232.501 ± 8.320 | iters≈ 329 | lr=0.0515 depth=4 |  11.0s


[58/60] MSE = 232.189 ± 8.042 | iters≈ 476 | lr=0.0408 depth=4 |  13.8s


[59/60] MSE = 223.756 ± 7.484 | iters≈ 296 | lr=0.0316 depth=7 |  19.7s


[60/60] MSE = 230.492 ± 5.170 | iters≈  72 | lr=0.0847 depth=8 |  12.8s

Search finished in 13.4 min
Best CV MSE: 223.322
Best config: {'n_estimators': 3000, 'learning_rate': 0.023818871525191507, 'max_depth': 7, 'min_child_weight': 8, 'subsample': 0.7343701351517025, 'colsample_bytree': 0.8478267904075705, 'gamma': 0.29079784571475303, 'reg_alpha': 0.11872892198703458, 'reg_lambda': 0.6407829541336535}


,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,mse_mean,mse_std,avg_best_iter,seconds
0,3000,0.023819,7,8,0.734370,0.847827,0.290798,0.118729,0.640783,223.321784,7.032011,316,15.368394
1,3000,0.025637,7,8,0.873429,0.940253,0.130330,0.009997,1.474096,223.729206,6.376054,377,17.074955
2,3000,0.023308,7,8,0.675348,0.930782,0.054963,0.387248,3.161216,223.736009,6.354437,411,25.990690
3,3000,0.031584,7,10,0.681926,0.949731,0.266274,0.056842,0.615759,223.756290,7.483779,296,19.681853
4,3000,0.024836,8,6,0.804267,0.907272,0.185120,0.010948,2.180385,223.914930,7.097071,375,32.862511
5,3000,0.024408,6,11,0.769273,0.740285,0.195434,0.061525,4.514330,223.952988,7.713384,401,15.384774
6,3000,0.031714,8,11,0.657458,0.816559,0.253590,0.001931,0.690732,224.259923,6.623191,264,18.175552
7,3000,0.034009,7,2,0.681021,0.826293,0.068237,0.313958,1.905597,224.439187,6.678208,325,17.013671
8,3000,0.030912,7,5,0.744332,0.697283,0.059113,0.336186,1.370484,224.632964,6.678495,285,19.118747
9,3000,0.020582,7,2,0.898829,0.889045,0.093056,0.027073,2.018300,224.776920,5.925880,443,24.006705


## 6. Final 5-Fold OOF with Best Config
Re-evaluate the winner to also extract OOF predictions and the per-fold trained models (for the test-set inference).

In [ ]:
final_res = evaluate_config(best_cfg, return_oof=True, return_models=True, verbose=True)
print(f"\nXGBoost v2 | OOF MSE = {final_res['mse_mean']:.3f} ± {final_res['mse_std']:.3f}")
np.save(OUT_DIR / 'oof_XGBoostV2.npy', final_res['oof'])

    fold 1: MSE=214.718, best_iter=350


    fold 2: MSE=233.445, best_iter=348


    fold 3: MSE=228.320, best_iter=269


    fold 4: MSE=216.561, best_iter=303


    fold 5: MSE=223.565, best_iter=314

XGBoost v2 | OOF MSE = 223.322 ± 7.032


## 7. Test-Set Prediction — Average the 5 Fold-Models
Each fold's model used early stopping → averaging them is more robust than refitting on full data with a fixed iter count.

In [ ]:
test_preds = np.zeros(len(X_test))
for model, pp, best_iter in final_res['models']:
    X_test_proc = pp.transform(X_test)
    p = np.expm1(model.predict(X_test_proc, iteration_range=(0, best_iter + 1)))
    test_preds += np.clip(p, 0, 92)
test_preds /= len(final_res['models'])
test_preds_int = np.clip(np.round(test_preds), 0, 92).astype(int)
print('Test preds:', test_preds_int.min(), '→', test_preds_int.max(),
      '| mean =', test_preds_int.mean().round(2))

Test preds: 0 → 87 | mean = 13.71


In [ ]:
# Write submission with the byte-safe pattern that worked for Kaggle
lines = [b'PropertyID_test,Pred\n']
for i, p in zip(test_ids, test_preds_int):
    lines.append(f'{int(i)},{int(p)}\n'.encode('ascii'))
(OUT_DIR / 'submission_xgboost_v2.csv').write_bytes(b''.join(lines))

chk = pd.read_csv(OUT_DIR / 'submission_xgboost_v2.csv')
print('Rows:', len(chk), '| NaN:', chk.isna().sum().sum(), '| header:', list(chk.columns))
chk.head()

Rows: 24318 | NaN: 0 | header: ['PropertyID_test', 'Pred']


,PropertyID_test,Pred
0,795,0
1,2515,45
2,2595,2
3,5099,29
4,5107,21


## 8. Top-20 Search Configs (for the paper)

In [ ]:
show_cols = ['mse_mean', 'mse_std', 'learning_rate', 'max_depth', 'min_child_weight',
             'subsample', 'colsample_bytree', 'gamma', 'reg_alpha', 'reg_lambda',
             'avg_best_iter', 'seconds']
log_df.head(20)[show_cols]

,mse_mean,mse_std,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,avg_best_iter,seconds
0,223.321784,7.032011,0.023819,7,8,0.734370,0.847827,0.290798,0.118729,0.640783,316,15.368394
1,223.729206,6.376054,0.025637,7,8,0.873429,0.940253,0.130330,0.009997,1.474096,377,17.074955
2,223.736009,6.354437,0.023308,7,8,0.675348,0.930782,0.054963,0.387248,3.161216,411,25.990690
3,223.756290,7.483779,0.031584,7,10,0.681926,0.949731,0.266274,0.056842,0.615759,296,19.681853
4,223.914930,7.097071,0.024836,8,6,0.804267,0.907272,0.185120,0.010948,2.180385,375,32.862511
5,223.952988,7.713384,0.024408,6,11,0.769273,0.740285,0.195434,0.061525,4.514330,401,15.384774
6,224.259923,6.623191,0.031714,8,11,0.657458,0.816559,0.253590,0.001931,0.690732,264,18.175552
7,224.439187,6.678208,0.034009,7,2,0.681021,0.826293,0.068237,0.313958,1.905597,325,17.013671
8,224.632964,6.678495,0.030912,7,5,0.744332,0.697283,0.059113,0.336186,1.370484,285,19.118747
9,224.776920,5.925880,0.020582,7,2,0.898829,0.889045,0.093056,0.027073,2.018300,443,24.006705


## Summary
- Hand-rolled CV with **fold-level early stopping** gave each config a fair shot.
- 60 random configs explored, best logged to `xgb_v2_search_log.csv`.
- New OOF saved as `oof_XGBoostV2.npy` — drop into notebook 06 to rebuild the blend.
- New test submission `submission_xgboost_v2.csv` ready (header `PropertyID_test,Pred`, integers, byte-safe writer).

Next step: re-run notebook **06** to rebuild the blend with the improved XGBoost OOF.